In [ ]:
!nvidia-smi

Mon Jul 13 01:05:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
!git clone --branch asal/CiriEXT4 https://github.com/isusbu/CIRI-FS.git
%cd /content/CIRI-FS

Cloning into 'CIRI-FS'...
remote: Enumerating objects: 19101, done.
remote: Counting objects: 100% (19101/19101), done.
remote: Compressing objects: 100% (10372/10372), done.
remote: Total 19101 (delta 8737), reused 19042 (delta 8711), pack-reused 0 (from 0)
Receiving objects: 100% (19101/19101), 5.99 MiB | 18.08 MiB/s, done.
Resolving deltas: 100% (8737/8737), done.
/content/CIRI-FS


In [2]:
!git branch --show-current

asal/CiriEXT4


In [3]:
!pip install -q -r requirements.txt
!pip install -q --upgrade "transformers>=4.45.0" accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.7 MB/s eta 0:00:00


In [4]:
%cd /content/CIRI-FS

/content/CIRI-FS


In [5]:
import torch
import transformers
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch: 2.11.0+cu128
Transformers: 5.13.1
CUDA available: True
GPU: Tesla T4


In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-Coder-7B-Instruct"
)

print("Tokenizer loaded successfully")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded successfully


In [7]:
from pathlib import Path

path = Path("ciri/query/llm_gen.py")
text = path.read_text()

if "class QwenGen" not in text:
    text += '''

class QwenGen(BaseGen):
    def __init__(self, args: Dict, config_file: str, model, tokenizer):
        super().__init__(args, config_file)
        self.llm_model = model
        self.tokenizer = tokenizer
        self.device = get_device()

    def _generate(self) -> List:
        message = f"{self.config_file}\\n{self.prompt}"

        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": message}
        ]

        formatted = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(
            formatted,
            return_tensors="pt"
        ).to(self.device)

        input_len = inputs["input_ids"].shape[1]

        with torch.inference_mode():
            outputs = self.llm_model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=True,
                temperature=0.2,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )

        answer_list = []
        for output in outputs:
            generated = output[input_len:]
            answer_list.append(
                self.tokenizer.decode(
                    generated,
                    skip_special_tokens=True
                ).strip()
            )

        return answer_list
'''
    path.write_text(text)
    print("QwenGen added")
else:
    print("QwenGen already exists")

QwenGen added


In [8]:
from pathlib import Path

path = Path("ciri/ciri_runner.py")
text = path.read_text()

text = text.replace(
    "from ciri.query.llm_gen import GPTGen, ClaudeGen, LlamaGen, DeepseekGen",
    "from ciri.query.llm_gen import GPTGen, ClaudeGen, LlamaGen, DeepseekGen, QwenGen"
)

old = '''    elif args.model.startswith("deepseek"):
        return DeepseekGen(args, file_content, model, tokenizer)
    else:'''

new = '''    elif args.model.startswith("deepseek"):
        return DeepseekGen(args, file_content, model, tokenizer)
    elif args.model.startswith("Qwen"):
        return QwenGen(args, file_content, model, tokenizer)
    else:'''

if old in text:
    text = text.replace(old, new)

path.write_text(text)
print("ciri_runner.py updated")

ciri_runner.py updated


In [9]:
from pathlib import Path

path = Path("ciri/ciri_eng.py")
text = path.read_text()

text = text.replace(
    '"deepseek-coder-6.7b-instruct"],',
    '"deepseek-coder-6.7b-instruct", "Qwen2.5-Coder-7B-Instruct"],'
)

marker = '''        elif checkpoint.startswith("CodeLLaMa"):'''

qwen_loader = '''        elif checkpoint.startswith("Qwen"):
            from transformers import BitsAndBytesConfig

            full_checkpoint = "Qwen/Qwen2.5-Coder-7B-Instruct"

            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True
            )

            model = AutoModelForCausalLM.from_pretrained(
                full_checkpoint,
                quantization_config=quantization_config,
                device_map="auto",
                trust_remote_code=True
            )

            tokenizer = AutoTokenizer.from_pretrained(
                full_checkpoint,
                trust_remote_code=True,
                padding_side="left"
            )

            if tokenizer.pad_token_id is None:
                tokenizer.pad_token = tokenizer.eos_token

'''

if 'checkpoint.startswith("Qwen")' not in text:
    text = text.replace(marker, qwen_loader + marker)

text = text.replace('        model = model.to("cuda")\n', '')

path.write_text(text)
print("ciri_eng.py updated")

ciri_eng.py updated


In [10]:
!python -m py_compile \
  ciri/query/llm_gen.py \
  ciri/ciri_runner.py \
  ciri/ciri_eng.py

In [11]:
!python -m ciri.ciri_eng --help

usage: ciri_eng.py [-h] --input_path INPUT_PATH --output_path OUTPUT_PATH
                   --model
                   {gpt-3.5-turbo-0125,gpt-4-0125-preview,claude-3-opus-20240228,claude-3-sonnet-20240229,CodeLLaMa-7b-Instruct-hf,CodeLLaMa-13b-Instruct-hf,CodeLLaMa-34b-Instruct-hf,deepseek-coder-6.7b-instruct}
                   --system SYSTEM --version VERSION
                   [--validconfig_shot_num {0,1,2,3,4,5}]
                   [--misconfig_shot_num {0,1,2,3,4,5}]
                   [--shot_selection {random,similarity}]
                   [--file_format {xml,yaml,properties,conf}]
                   [--language {java,python,cpp}] [--read_code]
                   [--read_code_loc READ_CODE_LOC] [--shot_system SHOT_SYSTEM]
                   [--verbose]

Ciri Engine - Configuration Analysis Tool

options:
  -h, --help            show this help message and exit
  --validconfig_shot_num {0,1,2,3,4,5}
                        Number of valid configuration shots (default: 1)
  --

In [12]:
!grep -n "Qwen" ciri/ciri_eng.py

137:        elif checkpoint.startswith("Qwen"):
140:            full_checkpoint = "Qwen/Qwen2.5-Coder-7B-Instruct"


In [13]:
!grep -n "Qwen" ciri/query/llm_gen.py

180:class QwenGen(BaseGen):


In [14]:
!grep -n "Qwen" ciri/ciri_runner.py

5:from ciri.query.llm_gen import GPTGen, ClaudeGen, LlamaGen, DeepseekGen, QwenGen
25:    elif args.model.startswith("Qwen"):
26:        return QwenGen(args, file_content, model, tokenizer)


In [15]:
from pathlib import Path

path = Path("ciri/ciri_eng.py")
text = path.read_text()

old = '''                                  "deepseek-coder-6.7b-instruct"],'''

new = '''                                  "deepseek-coder-6.7b-instruct",
                                  "Qwen2.5-Coder-7B-Instruct"],'''

if old not in text:
    print("Exact text not found. Showing nearby lines:")
    for i, line in enumerate(text.splitlines(), 1):
        if "deepseek-coder-6.7b-instruct" in line:
            print(i, line)
else:
    text = text.replace(old, new)
    path.write_text(text)
    print("Qwen added to model choices.")

Exact text not found. Showing nearby lines:
305                              "deepseek-coder-6.7b-instruct"


In [16]:
!grep -n "deepseek-coder\|Qwen2.5" ciri/ciri_eng.py

140:            full_checkpoint = "Qwen/Qwen2.5-Coder-7B-Instruct"
305:                             "deepseek-coder-6.7b-instruct"


In [17]:
!python -m py_compile \
  ciri/query/llm_gen.py \
  ciri/ciri_runner.py \
  ciri/ciri_eng.py

In [18]:
!python -m ciri.ciri_eng --help

usage: ciri_eng.py [-h] --input_path INPUT_PATH --output_path OUTPUT_PATH
                   --model
                   {gpt-3.5-turbo-0125,gpt-4-0125-preview,claude-3-opus-20240228,claude-3-sonnet-20240229,CodeLLaMa-7b-Instruct-hf,CodeLLaMa-13b-Instruct-hf,CodeLLaMa-34b-Instruct-hf,deepseek-coder-6.7b-instruct}
                   --system SYSTEM --version VERSION
                   [--validconfig_shot_num {0,1,2,3,4,5}]
                   [--misconfig_shot_num {0,1,2,3,4,5}]
                   [--shot_selection {random,similarity}]
                   [--file_format {xml,yaml,properties,conf}]
                   [--language {java,python,cpp}] [--read_code]
                   [--read_code_loc READ_CODE_LOC] [--shot_system SHOT_SYSTEM]
                   [--verbose]

Ciri Engine - Configuration Analysis Tool

options:
  -h, --help            show this help message and exit
  --validconfig_shot_num {0,1,2,3,4,5}
                        Number of valid configuration shots (default: 1)
  --

In [19]:
from pathlib import Path
import re

path = Path("ciri/ciri_eng.py")
text = path.read_text()

pattern = r'("deepseek-coder-6\.7b-instruct")(\s*\])'

updated_text, count = re.subn(
    pattern,
    r'\1, "Qwen2.5-Coder-7B-Instruct"\2',
    text,
    count=1
)

if count == 0:
    print("Could not find the model choices entry.")
else:
    path.write_text(updated_text)
    print("Qwen added to the allowed model list.")

Qwen added to the allowed model list.


In [20]:
!grep -n -A4 -B4 "deepseek-coder-6.7b-instruct" ciri/ciri_eng.py

301-                             "claude-3-sonnet-20240229",
302-                             "CodeLLaMa-7b-Instruct-hf",
303-                             "CodeLLaMa-13b-Instruct-hf",
304-                             "CodeLLaMa-34b-Instruct-hf",
305:                             "deepseek-coder-6.7b-instruct", "Qwen2.5-Coder-7B-Instruct"
306-                         ],
307-                         help="Name of the model to use")
308-    required.add_argument("--system", required=True, type=str,
309-                         help="Software System name for processing")


In [21]:
!python -m py_compile ciri/ciri_eng.py
!python -m ciri.ciri_eng --help

usage: ciri_eng.py [-h] --input_path INPUT_PATH --output_path OUTPUT_PATH
                   --model
                   {gpt-3.5-turbo-0125,gpt-4-0125-preview,claude-3-opus-20240228,claude-3-sonnet-20240229,CodeLLaMa-7b-Instruct-hf,CodeLLaMa-13b-Instruct-hf,CodeLLaMa-34b-Instruct-hf,deepseek-coder-6.7b-instruct,Qwen2.5-Coder-7B-Instruct}
                   --system SYSTEM --version VERSION
                   [--validconfig_shot_num {0,1,2,3,4,5}]
                   [--misconfig_shot_num {0,1,2,3,4,5}]
                   [--shot_selection {random,similarity}]
                   [--file_format {xml,yaml,properties,conf}]
                   [--language {java,python,cpp}] [--read_code]
                   [--read_code_loc READ_CODE_LOC] [--shot_system SHOT_SYSTEM]
                   [--verbose]

Ciri Engine - Configuration Analysis Tool

options:
  -h, --help            show this help message and exit
  --validconfig_shot_num {0,1,2,3,4,5}
                        Number of valid configurati

In [22]:
!python -m py_compile \
  ciri/query/llm_gen.py \
  ciri/ciri_runner.py \
  ciri/ciri_eng.py

In [23]:
!python -m ciri.ciri_eng \
  --input_path icse25_data/datasets/synthesize_config/ext4_SD/erroneous/1 \
  --output_path /content/qwen_ext4_sd_test_1 \
  --model Qwen2.5-Coder-7B-Instruct \
  --system ext4 \
  --version 1.0 \
  --validconfig_shot_num 0 \
  --misconfig_shot_num 0 \
  --file_format xml \
  --verbose

2026-07-13 04:42:46 - Ciri - INFO - [llm_gen] Using device: CUDA
2026-07-13 04:42:46 - Ciri - ERROR - Ciri Engine failed: 'NoneType' object has no attribute 'apply_chat_template'
Traceback (most recent call last):
  File "/content/CIRI-FS/ciri/ciri_eng.py", line 364, in main
    process_files(args)
  File "/content/CIRI-FS/ciri/ciri_eng.py", line 271, in process_files
    process_single_file(args, input_path, output_path)
  File "/content/CIRI-FS/ciri/ciri_eng.py", line 235, in process_single_file
    ciri_runner(args, input_content, output_path, model, tokenizer)
  File "/content/CIRI-FS/ciri/ciri_runner.py", line 12, in ciri_runner
    result, reasons = _run_analysis(llm_gen)
                      ^^^^^^^^^^^^^^^^^^^^^^
  File "/content/CIRI-FS/ciri/ciri_runner.py", line 33, in _run_analysis
    answer_parser = llm_gen.generate()
                    ^^^^^^^^^^^^^^^^^^
  File "/content/CIRI-FS/ciri/query/llm_gen.py", line 50, in generate
    new_outputs = self._generate()
            

In [24]:
from pathlib import Path

path = Path("ciri/ciri_eng.py")
text = path.read_text()

old = '''    elif input_path.is_file():
        output_path.parent.mkdir(parents=True, exist_ok=True)
        if output_path.exists():
            output_path.unlink()
        process_single_file(args, input_path, output_path)
'''

new = '''    elif input_path.is_file():
        output_path.parent.mkdir(parents=True, exist_ok=True)
        if output_path.exists():
            output_path.unlink()

        model, tokenizer = load_language_model(args.model)
        update_logger_handler(output_path)
        process_single_file(
            args,
            input_path,
            output_path,
            model,
            tokenizer
        )
'''

if old not in text:
    print("Target block not found.")
else:
    path.write_text(text.replace(old, new))
    print("Single-file model loading fixed.")

Single-file model loading fixed.


In [25]:
!python -m py_compile ciri/ciri_eng.py

In [26]:
!python -m ciri.ciri_eng \
  --input_path icse25_data/datasets/synthesize_config/ext4_SD/erroneous/1 \
  --output_path /content/qwen_ext4_sd_test_1 \
  --model Qwen2.5-Coder-7B-Instruct \
  --system ext4 \
  --version 1.0 \
  --validconfig_shot_num 0 \
  --misconfig_shot_num 0 \
  --file_format xml \
  --verbose

2026-07-13 04:47:12 - Ciri - INFO - Using device: CUDA
2026-07-13 04:47:12 - Ciri - INFO - Using dtype: torch.bfloat16
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 70.1MB/s]
Fetching 4 files: 100% 4/4 [06:47<00:00, 101.86s/it]
Download complete: 100% 15.2G/15.2G [06:47<00:00, 37.4MB/s]
Loading weights:   1% 2/339 [00:08<25:04,  4.47s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 339/339 [01:01<00:00,  5.48it/s]
generation_config.json: 100% 242/242 [00:00<00:00, 1.32MB/s]
2026-07-13 04:55:10 - Ciri - INFO - Model loaded successfully on CUDA!
2026-07-13 04:55:10 - Ciri - INFO - [llm_gen] Using device: CUDA
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTo

In [27]:
!cat icse25_data/datasets/synthesize_config/ext4_SD/erroneous/1

<?xml version="1.0"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>

<configuration>
<property>
  <name>s_first_meta_bg</name>
  <value>99999999</value>
  <description>Index of the first block group using the meta_bg layout.</description>
</property>

<property>
  <name>mke2fs.blocksize</name>
  <value>4096</value>
  <description>The block size of the filesystem in bytes.</description>
</property>

<property>
  <name>mke2fs.inode_size</name>
  <value>256</value>
  <description>The size of each inode in bytes.</description>
</property>
</configuration>


In [28]:
!head -1 icse25_data/datasets/synthesize_config/ground_truth/ext4_SD.tsv

Range	Basic Numeric	s_first_meta_bg	99999999


In [29]:
!cat /content/qwen_ext4_sd_test_1

Final result:

The CONFIGURATION FILE IS CORRECT


In [31]:
!grep -R "s_first_meta_bg" -n .

./ciri_debug.log:8:  <name>s_first_meta_bg</name>
./icse25_data/datasets/synthesize_config/ext4_SD/erroneous/1:6:  <name>s_first_meta_bg</name>
./icse25_data/datasets/synthesize_config/ext4_SD/erroneous/5:6:  <name>ext4.s_first_meta_bg</name>
./icse25_data/datasets/synthesize_config/ext4_SD/correct/1:6:  <name>s_first_meta_bg</name>
./icse25_data/datasets/synthesize_config/ext4_SD/correct/5:6:  <name>ext4.s_first_meta_bg</name>
./icse25_data/datasets/synthesize_config/ground_truth/ext4_SD.tsv:1:Range	Basic Numeric	s_first_meta_bg	99999999
./icse25_data/datasets/synthesize_config/ground_truth/ext4_SD.tsv:5:Range	Basic Numeric	ext4.s_first_meta_bg	65536
./icse25_data/results/synthesize_config/ext4_SD/gpt-3.5-turbo-0125/zero_shot/erroneous/1:3:There are 1 misconfiguration parameters in the input: s_first_meta_bg
./icse25_data/results/synthesize_config/ext4_SD/gpt-3.5-turbo-0125/zero_shot/erroneous/1:5:Reason for s_first_meta_bg: The property 's_first_meta_bg' is not a valid property for e

In [32]:
!python -m ciri.ciri_eng \
  --input_path icse25_data/datasets/synthesize_config/ext4_SD/erroneous \
  --output_path icse25_data/results/synthesize_config/ext4_SD/Qwen2.5-Coder-7B-Instruct/zero_shot/erroneous \
  --model Qwen2.5-Coder-7B-Instruct \
  --system ext4 \
  --version 1.47.0 \
  --validconfig_shot_num 0 \
  --misconfig_shot_num 0 \
  --file_format xml \
  --verbose

2026-07-13 05:10:16 - Ciri - INFO - Using device: CUDA
2026-07-13 05:10:16 - Ciri - INFO - Using dtype: torch.bfloat16
Loading weights:   1% 2/339 [00:08<24:31,  4.37s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 339/339 [01:01<00:00,  5.51it/s]
2026-07-13 05:11:25 - Ciri - INFO - Model loaded successfully on CUDA!
2026-07-13 05:11:25 - Ciri - INFO - [llm_gen] Using device: CUDA
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[Ciri] Start
[Ciri] Running for file icse25_data/results/synthesize_config/ext4_SD/Qwen2.5-Coder-7B-Instruct/zero_shot/erroneous/3
[Ci

In [33]:
!python -m ciri.ciri_eng \
  --input_path icse25_data/datasets/synthesize_config/ext4_SD/correct \
  --output_path icse25_data/results/synthesize_config/ext4_SD/Qwen2.5-Coder-7B-Instruct/zero_shot/correct \
  --model Qwen2.5-Coder-7B-Instruct \
  --system ext4 \
  --version 1.47.0 \
  --validconfig_shot_num 0 \
  --misconfig_shot_num 0 \
  --file_format xml \
  --verbose

2026-07-13 05:13:27 - Ciri - INFO - Using device: CUDA
2026-07-13 05:13:27 - Ciri - INFO - Using dtype: torch.bfloat16
Loading weights:   1% 2/339 [00:08<24:29,  4.36s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 339/339 [01:01<00:00,  5.48it/s]
2026-07-13 05:14:36 - Ciri - INFO - Model loaded successfully on CUDA!
2026-07-13 05:14:36 - Ciri - INFO - [llm_gen] Using device: CUDA
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[Ciri] Start
[Ciri] Running for file icse25_data/results/synthesize_config/ext4_SD/Qwen2.5-Coder-7B-Instruct/zero_shot/correct/3
[Ciri

In [34]:
!python icse25_data/script/result_parser.py \
  --project ext4_SD \
  --model Qwen2.5-Coder-7B-Instruct \
  --mode zero_shot

[Ciri Result] on ext4_SD with Qwen2.5-Coder-7B-Instruct and zero_shot mode
File-Level: Precision: 1.00, Recall: 0.20, Accuracy: 0.60, F1: 0.33
Param-Level: Precision: 1.00, Recall: 0.20, Accuracy: 0.95, F1: 0.33
